# Data Exploration and Quality Analysis

## Objetivo

Este notebook tem como objetivo realizar análises exploratórias dos dados gerados ao longo do pipeline, permitindo validar a qualidade das informações, monitorar regras de negócio, investigar registros rejeitados e avaliar os indicadores produzidos nas camadas Silver e Gold.

O notebook serve como apoio ao desenvolvimento, testes e validação do projeto de Engenharia de Dados.

In [0]:
%run ../config/feat_squad2_config_adls

In [0]:
%run ../utils/feat_squad2_utils

In [0]:
path_quarantine_clientes = f"abfs://{container_name_data_lake}/quarantine/ecommerce_clientes"
path_quarantine_enderecos = f"abfs://{container_name_data_lake}/quarantine/ecommerce_enderecos"
path_silver_clientes = f"abfs://{container_name_data_lake}/silver/ecommerce_clientes"
path_silver_enderecos = f"abfs://{container_name_data_lake}/silver/ecommerce_enderecos"

storage_options = {
    "storage_account_name": storage_account_name,
    "tenant_id": tenant_id,
    "client_id": client_id,
    "client_secret": client_secret
}

In [0]:
try:
    df_silver_clientes = read_delta_to_spark_df(
        path=path_silver_clientes,
        storage_options=storage_options
    )
    print("Silver Clientes carregado com sucesso.")
except:
    print("Silver Clientes não foi carregada.")

try:
    df_silver_enderecos = read_delta_to_spark_df(
        path=path_silver_enderecos,
        storage_options=storage_options
    )
    print("Silver Enderecos carregado com sucesso.")
except:
    print("Silver Enderecos não foi carregada.")

try:
    df_quarantine_clientes = read_delta_to_spark_df(
        path=path_quarantine_clientes,
        storage_options=storage_options
    )
    print("Quarantine Clientes carregado com sucesso.")
except:
    print("Quarantine Clientes não foi carregada.")

try:
    df_silver_enderecos = read_delta_to_spark_df(
        path=path_silver_enderecos,
        storage_options=storage_options
    )
    print("Silver Enderecos carregado com sucesso.")
except:
    print("Silver Enderecos não foi carregada.")


In [0]:
display(df_silver_clientes)

In [0]:
display(df_silver_enderecos)

In [0]:
display(df_quarantine_clientes)

In [0]:
display(df_quarantine_enderecos)

In [0]:
from pyspark.sql.functions import count

df_chart = (
    df_quarantine_enderecos
    .groupBy("quarantine_reason")
    .agg(
        count("*").alias("qtd_registros")
    )
    .orderBy("qtd_registros", ascending=False)
)

display(df_chart)

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import lit

df_counts = (
    df_quarantine_enderecos.select(lit("quarantine").alias("source"), count("*").alias("qtde_registros"))
    .unionByName(
        df_silver_enderecos.select(lit("silver").alias("source"), count("*").alias("qtde_registros"))
    )
)

display(df_counts, "pie")

Databricks visualization. Run in Databricks to view.